# 02 - Baselines: stored numbers and a reproduction check

The stored results in `data/results/` come from `bash .runs/run_all.sh`-style runs (see
`scripts/reproduce.sh`). This notebook tabulates them, plots them, and re-evaluates two baselines on
a few held-out seeds to show that the stored numbers reproduce exactly (the metrics are
deterministic given scenario, seed and policy).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, functools
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from rl_flatland.plotting import load_rows, plot_results, plot_training, POLICY_LABEL
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
rows = load_rows(ROOT / "data" / "results")
df = pd.DataFrame(rows)
print(len(df), "episodes;", sorted(df.policy.unique()))

In [ ]:
summary = (df.groupby(["policy", "split", "scenario"])
             [["arrival_rate", "normalized_reward", "deadlocked", "decision_ms_mean", "setup_s"]]
             .mean().round(3))
summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
plot_results(rows, "arrival_rate", "test", ax=axes[0], ylabel="arrival rate (no malfunctions)")
plot_results(rows, "arrival_rate", "test_malfunction", ax=axes[1], ylabel="arrival rate (malfunctions)")
axes[1].get_legend().remove()
plt.tight_layout()

In [ ]:
logs = {name: ROOT / "data" / "checkpoints" / name / "train_log.jsonl" for name in ["ppo", "bc_ppo", "ppo_tree"]}
logs = {k: v for k, v in logs.items() if v.exists()}
plot_training(logs);

## Reproduce a slice of the stored numbers

Re-run the OR reference and PPO on the first three held-out seeds of the small scenario and
compare with the stored rows. Arrival rate, normalised reward and deadlocks must match exactly;
decision times will not (they depend on the machine and its load).

In [ ]:
import sys; sys.path.insert(0, str(ROOT / "scripts"))
from evaluate import make_policy
from rl_flatland.evaluation import evaluate
fresh = []
fresh += evaluate(functools.partial(make_policy, "or"), ["small"], ["test", "test_malfunction"], n_seeds=3, workers=1)
ckpt = ROOT / "data" / "checkpoints" / "ppo" / "model.pt"
if ckpt.exists():
    fresh += evaluate(functools.partial(make_policy, "network", str(ckpt), "compact", "ppo"), ["small"], ["test", "test_malfunction"], n_seeds=3, workers=1)
stored = {(r["policy"], r["scenario"], r["split"], r["seed"]): r for r in rows}
check = pd.DataFrame([dict(policy=r["policy"], split=r["split"], seed=r["seed"],
                           arrival_fresh=r["arrival_rate"], arrival_stored=stored[(r["policy"], r["scenario"], r["split"], r["seed"])]["arrival_rate"],
                           reward_fresh=round(r["normalized_reward"], 4), reward_stored=round(stored[(r["policy"], r["scenario"], r["split"], r["seed"])]["normalized_reward"], 4))
                      for r in fresh])
check["match"] = (check.arrival_fresh == check.arrival_stored) & (check.reward_fresh == check.reward_stored)
check